# L16 · 数据库实战：让数据持久化

**学习目标**
- 理解「数据库」为什么比内存列表可靠
- 用 `sqlite3`（Python 自带）建表、增、查
- 把 API 接到数据库，实现「重启也不丢数据」

**前置依赖**：L14、L15（FastAPI + 参数）  
**预计时长**：45 分钟  
**技术栈**：`sqlite3`(标准库)、`fastapi`、`uvicorn`、`requests`

---

## 概念讲解：数据库 = 永远记得住的账本

前面课程的服务，一重启数据就没了（因为存在内存里）。
**数据库** 是把数据写进硬盘的「账本」，关掉电脑再开，数据还在。

**SQLite** 是一个「零配置、单文件」的数据库，Python 自带，新手最友好。
它的语言叫 **SQL**：`CREATE`（建表）、`INSERT`（插）、`SELECT`（查）。

## 第一步：建表 + 插入（SQL 初见）

In [ ]:
import sqlite3
conn = sqlite3.connect("pets.db")
c = conn.cursor()
c.execute("CREATE TABLE IF NOT EXISTS pets (id INTEGER PRIMARY KEY, name TEXT, species TEXT)")
c.execute("INSERT OR IGNORE INTO pets (id, name, species) VALUES (1,'咪咪','猫'),(2,'旺财','狗')")
conn.commit()
print("✅ 已创建 pets 表并插入示例数据")

## 第二步：查询

In [ ]:
c.execute("SELECT * FROM pets")
for row in c.fetchall():
    print(row)
conn.close()

## 第三步：把 API 接到数据库

In [ ]:
from fastapi import FastAPI
import sqlite3, uvicorn, threading, time, requests

app = FastAPI()
def get_conn():
    conn = sqlite3.connect("pets.db")
    conn.row_factory = sqlite3.Row
    return conn

@app.get("/pets")
def list_pets():
    conn = get_conn()
    rows = conn.execute("SELECT * FROM pets").fetchall()
    conn.close()
    return [dict(r) for r in rows]

@app.post("/pets/{name}")
def add_pet(name: str, species: str = "未知"):
    conn = get_conn()
    conn.execute("INSERT INTO pets (name, species) VALUES (?,?)", (name, species))
    conn.commit(); conn.close()
    return {"ok": True, "added": name}

PORT = 8773
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"), daemon=True).start()
time.sleep(2)
print("✅ 数据库 API 已上线")

# 🎯 AHA 顿悟单元格：你的数据「重启也不丢」

运行下面代码：① 加一只新宠物 → ② 查列表确认它在 → ③ **即使你重跑整个 notebook 的「重启」**，
数据仍在 `pets.db` 文件里。这就是持久化的魔力。

> 真实世界的用户、订单、模型记录，全都活在数据库里。你今天亲手打通了「API ↔ 数据库」这条工业主干道。

In [ ]:
# ===== 运行我！（需先运行上面服务）=====
import requests, json
# ① 新增
r1 = requests.post(f"http://127.0.0.1:{PORT}/pets/吱吱", params={"species": "仓鼠"})
print("  新增：", r1.json())
# ② 查询
r2 = requests.get(f"http://127.0.0.1:{PORT}/pets")
print("  当前所有宠物：")
for p in r2.json():
    print("   ", p)
print("  💾 这些数据已写入 pets.db 文件 —— 重启 notebook 也还在！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：SQL 语法；`?` 占位符防注入（务必用参数化，不能 f-string 拼 SQL）。  
**易错点**：忘记 `commit()` 数据不落盘；`row_factory` 让结果变字典。  
**AHA 机制**：持久化到文件，重启不丢，强「工业级」实感。  
**衔接**：L17 鉴权（加 users 表）；L18 部署。  
**依赖**：标准库 sqlite3 + fastapi/uvicorn/requests。  
**清理**：可加 `import os; os.remove('pets.db')` 在测试后清理，但保留更利于学员体验。  
**注意**：多次运行会重复 insert 主键冲突，`INSERT OR IGNORE` 已处理；新增接口用自增 id 防止重复。

# 📚 作业 / 下一步

1. 用 DB Browser 或 `sqlite3 pets.db "SELECT * FROM pets;"` 查看真实文件。
2. 加一个 `DELETE /pets/{id}` 接口删除宠物。
3. 下一课 **L17 用户鉴权：守住你的服务大门** —— 给 API 加上「账号密码」守卫。